# Semana 4 — Busca Adversária: Minimax, α-β e MCTS
## INF0415 — Heurísticas e Modelagem Multiobjetivo | UFG — BIA

**Objetivo:** Implementar Minimax e Alfa-Beta para Jogo da Velha e Connect Four.

**Avaliação:** TL supervisionada (10 pts). Inclui torneio!

---

## 1. Classe TicTacToe (fornecida)

In [1]:
import random, time, copy
from typing import List, Optional, Tuple

class TicTacToe:
    def __init__(self):
        self.board = [' ']*9  # 0-8, ' '=vazio
        self.current = 'X'    # X=MAX, O=MIN
    
    def clone(self):
        g = TicTacToe()
        g.board = self.board[:]
        g.current = self.current
        return g
    
    def get_moves(self):
        return [i for i in range(9) if self.board[i]==' ']
    
    def make_move(self, pos):
        g = self.clone()
        g.board[pos] = g.current
        g.current = 'O' if g.current=='X' else 'X'
        return g
    
    def check_winner(self):
        lines = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
        for a,b,c in lines:
            if self.board[a]==self.board[b]==self.board[c]!=' ':
                return self.board[a]
        return None
    
    def is_terminal(self):
        return self.check_winner() is not None or len(self.get_moves())==0
    
    def utility(self):
        w = self.check_winner()
        if w=='X': return 1
        if w=='O': return -1
        return 0
    
    def display(self):
        b = self.board
        for i in range(3):
            row = [b[i*3+j] if b[i*3+j]!=' ' else '.' for j in range(3)]
            print(' | '.join(row))
            if i<2: print('---------')
        print(f'Vez: {self.current}\n')

print('✅ TicTacToe carregado!')
TicTacToe().display()

✅ TicTacToe carregado!
. | . | .
---------
. | . | .
---------
. | . | .
Vez: X



## 2. Minimax para Jogo da Velha 🔧

### ✏️ IMPLEMENTEM AQUI

**Algoritmo recursivo:**
- Se terminal: retornar utility
- Se MAX (X): retornar max dos filhos
- Se MIN (O): retornar min dos filhos

In [3]:
def minimax(state: TicTacToe, is_maximizing: bool) -> int:
    """Retorna o valor minimax do estado."""
    if state.is_terminal():
        return state.utility()
    
    if is_maximizing:
        value = -float('inf')
        for move in state.get_moves():
            child = state.make_move(move)
            value = max(value, minimax(child, False))
        return value
    else:
        value = float('inf')
        for move in state.get_moves():
            child = state.make_move(move)
            value = min(value, minimax(child, True))
        return value


def best_move_minimax(state: TicTacToe) -> int:
    """Retorna a melhor jogada para o jogador atual."""
    is_max = (state.current == 'X')
    best_val = -float('inf') if is_max else float('inf')
    best_m = None
    for move in state.get_moves():
        child = state.make_move(move)
        val = minimax(child, not is_max)
        if is_max and val > best_val:
            best_val = val; best_m = move
        elif not is_max and val < best_val:
            best_val = val; best_m = move
    return best_m

### Teste do Minimax

In [4]:
# Teste: minimax deve encontrar empate perfeito
g = TicTacToe()
t0 = time.time()
val = minimax(g, True)
dt = time.time()-t0
print(f'Valor minimax do estado inicial: {val}')
print(f'(0 = empate perfeito ✔)' if val==0 else '❌ ERRO')
print(f'Tempo: {dt:.3f}s')

Valor minimax do estado inicial: 0
(0 = empate perfeito ✔)
Tempo: 0.740s


## 3. Poda Alfa-Beta 🔧

### ✏️ IMPLEMENTEM AQUI

Adicionar α e β ao Minimax. Podar quando α ≥ β.

In [5]:
nodes_ab = 0  # contador global

def alphabeta(state, depth, alpha, beta, is_max):
    """Minimax com poda Alfa-Beta. Retorna valor."""
    global nodes_ab
    nodes_ab += 1
    
    if state.is_terminal():
        return state.utility()
    
    if is_max:
        value = -float('inf')
        for move in state.get_moves():
            child = state.make_move(move)
            value = max(value, alphabeta(child, depth+1, alpha, beta, False))
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return value
    else:
        value = float('inf')
        for move in state.get_moves():
            child = state.make_move(move)
            value = min(value, alphabeta(child, depth+1, alpha, beta, True))
            beta = min(beta, value)
            if alpha >= beta:
                break
        return value


def best_move_ab(state):
    global nodes_ab
    nodes_ab = 0
    is_max = (state.current=='X')
    best_val = -float('inf') if is_max else float('inf')
    best_m = None
    for move in state.get_moves():
        child = state.make_move(move)
        val = alphabeta(child, 0, -float('inf'), float('inf'), not is_max)
        if is_max and val>best_val: best_val=val; best_m=move
        elif not is_max and val<best_val: best_val=val; best_m=move
    return best_m

### Comparação: Minimax vs Alfa-Beta (nós expandidos)

In [6]:
# Contar nós do minimax puro
nodes_mm = 0
def minimax_count(state, is_max):
    global nodes_mm; nodes_mm += 1
    if state.is_terminal(): return state.utility()
    if is_max:
        v = -float('inf')
        for m in state.get_moves():
            v = max(v, minimax_count(state.make_move(m), False))
        return v
    else:
        v = float('inf')
        for m in state.get_moves():
            v = min(v, minimax_count(state.make_move(m), True))
        return v

g = TicTacToe()
nodes_mm = 0
minimax_count(g, True)
nodes_ab = 0
alphabeta(g, 0, -float('inf'), float('inf'), True)
print(f'Minimax puro:  {nodes_mm:>10,} nós')
print(f'Alfa-Beta:     {nodes_ab:>10,} nós')
print(f'Redução:       {100*(1-nodes_ab/nodes_mm):.1f}%')

Minimax puro:     549,946 nós
Alfa-Beta:         18,297 nós
Redução:       96.7%


## 4. Classe ConnectFour (fornecida)

In [7]:
class ConnectFour:
    ROWS, COLS = 6, 7
    def __init__(self):
        self.board = [['.']*self.COLS for _ in range(self.ROWS)]
        self.current = 'X'  # X=MAX, O=MIN
    def clone(self):
        g = ConnectFour()
        g.board = [row[:] for row in self.board]
        g.current = self.current
        return g
    def get_moves(self):
        return [c for c in range(self.COLS) if self.board[0][c]=='.']
    def make_move(self, col):
        g = self.clone()
        for r in range(self.ROWS-1, -1, -1):
            if g.board[r][col]=='.':
                g.board[r][col] = g.current
                break
        g.current = 'O' if g.current=='X' else 'X'
        return g
    def check_winner(self):
        b = self.board
        for r in range(self.ROWS):
            for c in range(self.COLS):
                if b[r][c]=='.': continue
                p = b[r][c]
                # Horizontal
                if c+3<self.COLS and all(b[r][c+i]==p for i in range(4)): return p
                # Vertical
                if r+3<self.ROWS and all(b[r+i][c]==p for i in range(4)): return p
                # Diag \\
                if r+3<self.ROWS and c+3<self.COLS and all(b[r+i][c+i]==p for i in range(4)): return p
                # Diag /
                if r+3<self.ROWS and c-3>=0 and all(b[r+i][c-i]==p for i in range(4)): return p
        return None
    def is_terminal(self):
        return self.check_winner() is not None or len(self.get_moves())==0
    def utility(self):
        w = self.check_winner()
        if w=='X': return 1
        if w=='O': return -1
        return 0
    def display(self):
        print('  '.join([str(i+1) for i in range(self.COLS)]))
        for row in self.board:
            print('  '.join(row))
        print(f'Vez: {self.current}\n')

print('✅ ConnectFour carregado!')
ConnectFour().display()

✅ ConnectFour carregado!
1  2  3  4  5  6  7
.  .  .  .  .  .  .
.  .  .  .  .  .  .
.  .  .  .  .  .  .
.  .  .  .  .  .  .
.  .  .  .  .  .  .
.  .  .  .  .  .  .
Vez: X



## 5. Função de Avaliação para Connect Four 🔧

### ✏️ IMPLEMENTEM AQUI

Projetem uma função que estima quem está ganhando.
Dicas: contar sequências de 2/3/4 peças, valorizar centro.

In [9]:
def evaluate_c4(state: ConnectFour, player: str) -> float:
    """
    Avalia o estado do Connect Four para `player`.
    Retorna score positivo = bom para player, negativo = ruim.
    """
    opp = 'O' if player == 'X' else 'X'
    score = 0

    # Pontuar centro
    center = [state.board[r][3] for r in range(6)]
    score += center.count(player) * 3

    # Avaliar todas as janelas de 4
    for r in range(6):
        for c in range(7):
            # Horizontal
            if c + 3 < 7:
                window = [state.board[r][c + i] for i in range(4)]
                score += score_window(window, player, opp)
            # Vertical
            if r + 3 < 6:
                window = [state.board[r + i][c] for i in range(4)]
                score += score_window(window, player, opp)
            # Diag \
            if r + 3 < 6 and c + 3 < 7:
                window = [state.board[r + i][c + i] for i in range(4)]
                score += score_window(window, player, opp)
            # Diag /
            if r + 3 < 6 and c - 3 >= 0:
                window = [state.board[r + i][c - i] for i in range(4)]
                score += score_window(window, player, opp)

    return score

def score_window(window, player, opp):
    """Pontua uma janela de 4 posições."""
    s = 0
    pc = window.count(player)
    oc = window.count(opp)
    ec = window.count('.')
    if pc==4: s += 1000
    elif pc==3 and ec==1: s += 50
    elif pc==2 and ec==2: s += 10
    if oc==3 and ec==1: s -= 80  # bloquear!
    return s

## 6. Minimax com Profundidade Limitada para Connect Four 🔧

### ✏️ IMPLEMENTEM AQUI

Adaptar Alfa-Beta para parar na profundidade d e usar evaluate_c4.

In [10]:
def alphabeta_c4(state, depth, alpha, beta, is_max, max_depth, eval_fn):
    if state.is_terminal():
        w = state.check_winner()
        if w=='X': return 100000
        if w=='O': return -100000
        return 0
    if depth >= max_depth:
        return eval_fn(state, 'X')  # avaliar do ponto de vista de MAX
    
    if is_max:
        value = float('-inf')
        for move in state.get_moves():
            child = state.make_move(move)
            value = max(value, alphabeta_c4(child, depth + 1, alpha, beta, False, max_depth, eval_fn))
            alpha = max(alpha, value)
            if alpha >= beta:
                break  # poda beta
        return value
    else:
        value = float('inf')
        for move in state.get_moves():
            child = state.make_move(move)
            value = min(value, alphabeta_c4(child, depth + 1, alpha, beta, True, max_depth, eval_fn))
            beta = min(beta, value)
            if alpha >= beta:
                break  # poda alfa
        return value
    
    return 0  # placeholder


def best_move_c4(state, max_depth=4, eval_fn=evaluate_c4):
    is_max = (state.current=='X')
    best_val = -float('inf') if is_max else float('inf')
    best_m = None
    for move in state.get_moves():
        child = state.make_move(move)
        val = alphabeta_c4(child, 0, -float('inf'), float('inf'), not is_max, max_depth, eval_fn)
        if is_max and val>best_val: best_val=val; best_m=move
        elif not is_max and val<best_val: best_val=val; best_m=move
    return best_m

## 7. Jogar e Testar

In [11]:
def play_game(game_class, agent1, agent2, verbose=False):
    """Joga uma partida. agent = função(state) -> move"""
    state = game_class()
    agents = {'X': agent1, 'O': agent2}
    while not state.is_terminal():
        move = agents[state.current](state)
        state = state.make_move(move)
        if verbose: state.display()
    return state.utility()  # +1=X wins, -1=O wins, 0=draw

def random_agent(state):
    return random.choice(state.get_moves())

# Teste: Minimax vs Random no Jogo da Velha
results = {'X':0, 'O':0, 'D':0}
for _ in range(100):
    r = play_game(TicTacToe, best_move_minimax, random_agent)
    if r==1: results['X']+=1
    elif r==-1: results['O']+=1
    else: results['D']+=1
print(f'Minimax(X) vs Random(O): {results}')
print(f'Minimax nunca perde!' if results['O']==0 else '❌ Algo errado')

Minimax(X) vs Random(O): {'X': 100, 'O': 0, 'D': 0}
Minimax nunca perde!


## 8. Torneio Connect Four 🏆

Submetam sua função evaluate_c4. Round-robin automático!

In [15]:
def tournament(eval_functions: dict, depth=4, rounds=10):
    """Torneio round-robin entre funções de avaliação."""
    names = list(eval_functions.keys())
    scores = {n: 0 for n in names}
    
    for i, n1 in enumerate(names):
        for j, n2 in enumerate(names):
            if i == j: continue
            wins = 0
            for _ in range(rounds):
                a1 = lambda s, ef=eval_functions[n1]: best_move_c4(s, depth, ef)
                a2 = lambda s, ef=eval_functions[n2]: best_move_c4(s, depth, ef)
                r = play_game(ConnectFour, a1, a2)
                if r == 1: wins += 1
                elif r == -1: wins -= 1
            scores[n1] += wins
    
    print('\n🏆 RANKING DO TORNEIO:')
    print('=' * 30)
    for i, (name, score) in enumerate(sorted(scores.items(), key=lambda x: -x[1])):
        medal = ['🥇', '🥈', '🥉'][i] if i < 3 else '  '
        print(f'{medal} {name:<20} {score:>+4}')

tournament({'MeuAgente': evaluate_c4, 'Random': lambda s,p: 0}, depth=4, rounds=50)

KeyboardInterrupt: 

## 9. Perguntas ✍️

### P1. O Minimax garante empate no Jogo da Velha? O que isso significa?
*Resposta:*

### P2. Quantos nós a menos Alfa-Beta expandiu vs Minimax puro? Essa redução foi boa?
*Resposta:*

### P3. Qual característica da sua função de avaliação para C4 fez mais diferença?
*Resposta:*

### P4. Profundidade 4 vs 6: o que mudou na qualidade das jogadas? E no tempo?
*Resposta:*

### P5. Como a função de avaliação em jogos se parece com h(n) do A*?
*Resposta:*

### P6. Por que MCTS funciona para Go mas Minimax não?
*Resposta:*

---
## 📚 Para as Semanas 5-6

**Busca Local:** Hill Climbing, Simulated Annealing, Busca Tabu.

Sem oponente, sem árvore — só vizinhança e fitness.

**Leitura:** Russell & Norvig, 4ª Ed., Cap. 4.1.

---
*INF0415 — UFG 2026/2*